<a href="https://colab.research.google.com/github/Shaheenovic/Drone-parking-monitoring-yolov8/blob/main/notebooks/02_training_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -q ultralytics pyyaml

In [35]:
!git clone https://github.com/Shaheenovic/Drone-parking-monitoring-yolov8.git
%cd Drone-parking-monitoring-yolov8
!git status

Cloning into 'Drone-parking-monitoring-yolov8'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 57 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 32.97 KiB | 3.66 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [36]:
from pathlib import Path
import urllib.request
import zipfile

OWNER = "Shaheenovic"
REPO = "Drone-parking-monitoring-yolov8"
TAG = "v1.0"
DATASET_ASSET = "drone-parking-v1-yolo11.zip"

url = f"https://github.com/{OWNER}/{REPO}/releases/download/{TAG}/{DATASET_ASSET}"

archive_path = Path("data") / DATASET_ASSET
extract_path = Path("data/raw")

archive_path.parent.mkdir(parents=True, exist_ok=True)
extract_path.mkdir(parents=True, exist_ok=True)

print("Downloading:", url)
urllib.request.urlretrieve(url, archive_path)

with zipfile.ZipFile(archive_path, "r") as z:
    z.extractall(extract_path)

print("Downloaded to:", archive_path)
print("Extracted to:", extract_path)

Downloading: https://github.com/Shaheenovic/Drone-parking-monitoring-yolov8/releases/download/v1.0/drone-parking-v1-yolo11.zip
Downloaded to: data/drone-parking-v1-yolo11.zip
Extracted to: data/raw


In [37]:
from pathlib import Path
import hashlib
import shutil
import yaml

EXPECTED_SHA256 = "3F9E38E8AE7F7725F19F5F120165A1F19E8A54808D27690648A530CFB6B12999".lower()

sha256 = hashlib.sha256()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256:  ", actual_sha256)

if actual_sha256 != EXPECTED_SHA256:
    raise ValueError("SHA256 mismatch: أعد تنزيل ملف الـdataset ولا تبدأ التدريب.")

print("SHA256 verified successfully.")

yaml_files = list(extract_path.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("لم يتم العثور على data.yaml بعد فك الضغط.")

print("\nFound data.yaml files:")
for p in yaml_files:
    print("-", p)

DATA_YAML = yaml_files[0].resolve()
DATASET_ROOT = DATA_YAML.parent.resolve()

print("\nDataset root:", DATASET_ROOT)
print("\nOriginal data.yaml:\n")
print(DATA_YAML.read_text())

Expected SHA256: 3f9e38e8ae7f7725f19f5f120165a1f19e8a54808d27690648a530cfb6b12999
Actual SHA256:   3f9e38e8ae7f7725f19f5f120165a1f19e8a54808d27690648a530cfb6b12999
SHA256 verified successfully.

Found data.yaml files:
- data/raw/data.yaml

Dataset root: /content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/data/raw

Original data.yaml:

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['Empty', 'Illegal', 'LicensePlate', 'Occupied']

roboflow:
  workspace: eng-ahmed_shaheen-hotmail-com
  project: drone-parking-monitoring-yolov8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/eng-ahmed_shaheen-hotmail-com/drone-parking-monitoring-yolov8/dataset/1


In [38]:
with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

data_config["path"] = str(DATASET_ROOT)

for split in ["train", "val", "test"]:
    if split in data_config:
        print(f"{split}: {data_config[split]}")

FIXED_DATA_YAML = Path("data") / "data_colab.yaml"

with open(FIXED_DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("\nSaved corrected config to:", FIXED_DATA_YAML)
print("\nCorrected data.yaml:\n")
print(FIXED_DATA_YAML.read_text())

train: ../train/images
val: ../valid/images
test: ../test/images

Saved corrected config to: data/data_colab.yaml

Corrected data.yaml:

train: ../train/images
val: ../valid/images
test: ../test/images
nc: 4
names:
- Empty
- Illegal
- LicensePlate
- Occupied
roboflow:
  workspace: eng-ahmed_shaheen-hotmail-com
  project: drone-parking-monitoring-yolov8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/eng-ahmed_shaheen-hotmail-com/drone-parking-monitoring-yolov8/dataset/1
path: /content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/data/raw



In [39]:
from pathlib import Path

for split in ["train", "valid", "val", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    if image_dir.exists():
        images = list(image_dir.glob("*.*"))
        labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

        print(f"{split}:")
        print(f"  Images: {len(images)}")
        print(f"  Labels: {len(labels)}")

train:
  Images: 401
  Labels: 401
valid:
  Images: 94
  Labels: 94


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(FIXED_DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="runs",
    name="drone_parking_yolov8n",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
    deterministic=True,
    patience=10,
    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.162 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.161 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, mome

In [ ]:
from pathlib import Path

run_dir = Path("/content/Drone-parking-monitoring-yolov8/Drone-parking-monitoring-yolov8/runs/detect/runs/drone_parking_yolov8n")
weights_dir = run_dir / "weights"
best_model_path = weights_dir / "best.pt"

print("Run directory:", run_dir)
print("Best model exists:", best_model_path.exists())
print("Best model path:", best_model_path)

best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(FIXED_DATA_YAML), split="val")

print(metrics)

for p in [
    run_dir / "results.csv",
    run_dir / "results.png",
    run_dir / "confusion_matrix.png",
    run_dir / "PR_curve.png",
    run_dir / "F1_curve.png",
    run_dir / "P_curve.png",
    run_dir / "R_curve.png",
    best_model_path,
]:
    print(p, "->", p.exists())

In [ ]:
from pathlib import Path
import shutil

repo_root = Path.cwd()
artifact_dir = repo_root / "artifacts" / "training"
artifact_dir.mkdir(parents=True, exist_ok=True)

patterns = [
    "**/weights/best.pt",
    "**/results.csv",
    "**/results.png",
    "**/confusion_matrix.png",
    "**/confusion_matrix_normalized.png",
    "**/PR_curve.png",
    "**/F1_curve.png",
    "**/P_curve.png",
    "**/R_curve.png",
]

found_files = []

for pattern in patterns:
    for source in repo_root.glob(pattern):
        if source.is_file() and "artifacts/training" not in str(source):
            destination = artifact_dir / source.name
            shutil.copy2(source, destination)
            found_files.append(destination)

print("Copied artifacts:")
for file_path in sorted(set(found_files)):
    print("-", file_path.relative_to(repo_root))

In [ ]:
for p in sorted(artifact_dir.iterdir()):
    print(f"{p.name:35} {p.stat().st_size / 1024:.1f} KB")

In [ ]:
!git config user.name "Shaheenovic"
!git config user.email "eng.ahmed_shaheen@hotmail.com"

!git config user.name
!git config user.email

In [ ]:
!git commit -m "Add YOLOv8 training results and best weights"